# 07 — Chạy code GRU gốc của tác giả

**Nguồn:** paper *Objective over Architecture* (2025), repo
[wn-nwN/fraud-detection-project-Nov2025](https://github.com/wn-nwN/fraud-detection-project-Nov2025).

Notebook clone đúng commit, thực thi các cell GRU gốc trong tiến trình
Python/IPython riêng và lưu output huấn luyện thật vào proof.

- Chạy 10 trial HPO, tổng 69 epoch, rồi fixed recipe 8 epoch: **77 epoch**.
- Giữ nguyên thứ tự RNG, Dataset, model, loss và optimizer của tác giả.
- Hai thay đổi có diff: đường dẫn CSV và phần ghi log từng trial.
- Phạm vi: GRU train/tune/test. Chưa gồm toàn bộ thí nghiệm của paper.

Dùng kernel `D:\UIT\.venvs\ml-gpu\Scripts\python.exe`.
Cần Git và requirements-author07.txt. Dự trù 1–2 giờ trên GPU, chưa đo pipeline này.


## 1. Import thư viện

Mỗi bước nằm trong một cell riêng. Các cell `%%writefile` bên dưới chỉ ghi
file hỗ trợ vào thư mục evidence của run; chúng không khởi chạy training.


In [1]:
from pathlib import Path
import contextlib
import datetime
import difflib
import hashlib
import importlib.metadata
import io
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
import nbformat
import torch


## 2. Project và cấu hình nguồn


In [2]:
PROJECT_DIR = None

here = Path.cwd().resolve()

ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (p for p in [here, *here.parents] if (p / "data/raw/Base.csv").is_file()), None
    )
)

if ROOT is None:
    raise FileNotFoundError("Mở BAF_project hoặc điền PROJECT_DIR.")

DATA_PATH = ROOT / "data/raw/Base.csv"

EXPECTED_ROWS = 1_000_000

EXPECTED_DATA_SHA256 = (
    "7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809"
)

REPO_URL = "https://github.com/wn-nwN/fraud-detection-project-Nov2025.git"

COMMIT = "6e8252cb9080378905b2aae30033c5946b72335b"

AUTHOR_FILE = "Binary_class_variable_analysis_github.ipynb"

AUTHOR_SHA256 = "5391e269bb5388ac5c0dde44662a950de1285e0f5f81c6b02315d466e3422e41"

REQUIRE_CUDA = True


### Hàm đọc và lưu bằng chứng


In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


### Kiểm tra môi trường và dữ liệu trước khi chạy


In [4]:
if shutil.which("git") is None:
    raise RuntimeError("Không thấy git trên PATH. Mở lại VS Code sau khi cài Git.")

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "Chọn kernel GPU đã chạy notebook06; không chạy77 epoch trên CPU."
    )

if sha256_file(DATA_PATH) != EXPECTED_DATA_SHA256:
    raise ValueError("Base.csv khác dữ liệu đã kiểm chứng.")

print("Project:", ROOT)

print("Python:", sys.executable)

print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Project: D:\UIT\DS317\BAF_project
Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe
Torch: 2.14.1+cu130 CUDA: 13.0
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 3. Tạo thư mục run và lấy code từ GitHub


In [5]:
suffix = datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]

RUN_DIR = ROOT / "results/runs" / ("author_gru_original_" + suffix)

RUN_DIR.mkdir(parents=True, exist_ok=False)

EVIDENCE = RUN_DIR / "evidence"

EVIDENCE.mkdir()

REPO_DIR = ROOT / "external" / ("gru_author_" + COMMIT[:12])

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)


### Hàm gọi Git và giữ log


In [6]:
def git_command(arguments, binary=False):
    result = subprocess.run(["git", *arguments], capture_output=True, check=False)
    with (EVIDENCE / "git.log").open("a", encoding="utf-8") as log:
        log.write("git " + " ".join(str(arg) for arg in arguments) + "\n")
        log.write(result.stderr.decode("utf-8", errors="replace") + "\n")
    if result.returncode:
        raise RuntimeError(result.stderr.decode("utf-8", errors="replace"))
    return result.stdout if binary else result.stdout.decode("utf-8").strip()


### Clone, checkout và kiểm tra commit


In [7]:
if not REPO_DIR.exists():
    git_command(["clone", "--no-checkout", REPO_URL, str(REPO_DIR)])
    git_command(["-C", str(REPO_DIR), "checkout", "--detach", COMMIT])

remote = git_command(["-C", str(REPO_DIR), "remote", "get-url", "origin"])

head = git_command(["-C", str(REPO_DIR), "rev-parse", "HEAD"])

dirty = git_command(["-C", str(REPO_DIR), "status", "--porcelain"])

if remote != REPO_URL or head != COMMIT or dirty:
    raise RuntimeError(
        "Repo không đúng remote/commit hoặc đang có thay đổi. Kiểm tra REPO_DIR."
    )

original_bytes = git_command(
    ["-C", str(REPO_DIR), "show", f"{COMMIT}:{AUTHOR_FILE}"], binary=True
)

if hashlib.sha256(original_bytes).hexdigest() != AUTHOR_SHA256:
    raise ValueError("SHA-256 notebook gốc không khớp.")


### Lưu notebook gốc và thông tin nguồn


In [8]:
(EVIDENCE / "author_original.ipynb").write_bytes(original_bytes)

original = json.loads(original_bytes)

for filename in ["LICENSE", "README.md"]:
    content = git_command(
        ["-C", str(REPO_DIR), "show", f"{COMMIT}:{filename}"], binary=True
    )
    (EVIDENCE / filename).write_bytes(content)

write_json(
    EVIDENCE / "git_identity.json",
    {
        "remote": remote,
        "commit": head,
        "working_tree_clean": not bool(dirty),
        "notebook_git_blob": git_command(
            ["-C", str(REPO_DIR), "rev-parse", f"{COMMIT}:{AUTHOR_FILE}"]
        ),
        "notebook_sha256": AUTHOR_SHA256,
    },
)

print("Clone/commit/source đã xác nhận:", head)

print("Run mới:", RUN_DIR)


Clone/commit/source đã xác nhận: 6e8252cb9080378905b2aae30033c5946b72335b
Run mới: D:\UIT\DS317\BAF_project\results\runs\author_gru_original_20261027_205457_a9c674ea


## 4. Chuẩn bị các file hỗ trợ cho proof

Các đoạn Python dưới đây được trình bày trực tiếp, không đặt trong chuỗi code dài.
`%%writefile` ghi phần đầu; `%%writefile -a` nối phần kế tiếp vào cùng file.
Các file này nằm trong evidence của lượt chạy, không thêm module vào src.


In [9]:
OBSERVER_FILE = (EVIDENCE / "observe_trials.py").as_posix()
POSTPROCESS_FILE = (EVIDENCE / "save_artifacts.py").as_posix()
RUNNER_FILE = (EVIDENCE / "execute_notebook.py").as_posix()
AUDIT_FILE = (EVIDENCE / "verify_results.py").as_posix()


### 4.1. Ghi log HPO

Gọi hàm train gốc và trả nguyên kết quả. Không lấy thêm minibatch hoặc gọi RNG.


In [10]:
%%writefile "{OBSERVER_FILE}"
_source_train_and_evaluate = train_and_evaluate
replay_trials = []


def _observed_train_and_evaluate(*args, **kwargs):
    trial_started = time.perf_counter()
    result = _source_train_and_evaluate(*args, **kwargs)
    params = args[0] if args else kwargs["hyperparams"]
    row = {
        "trial": len(replay_trials) + 1,
        "hyperparams": {
            key: value.item() if isinstance(value, np.generic) else value
            for key, value in params.items()
        },
        "validation_f1_weighted": float(result[0]),
        "validation_accuracy": float(result[1]),
        "seconds": time.perf_counter() - trial_started,
    }
    replay_trials.append(row)
    Path("hpo_trials.json").write_text(
        json.dumps(replay_trials, indent=2), encoding="utf-8"
    )
    print("HPO_TRIAL", json.dumps(row), flush=True)
    return result


train_and_evaluate = _observed_train_and_evaluate


Writing D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/observe_trials.py


### 4.2. Lưu kết quả sau cell test gốc

Tính lại metric, lưu predictions, split IDs, preprocessing và thông tin HPO.


In [11]:
%%writefile "{POSTPROCESS_FILE}"
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

scores = np.asarray(test_probabilities, dtype=np.float32)
labels = np.asarray(test_labels, dtype=np.int64)
predictions = np.asarray(test_predictions, dtype=np.int64)
if not np.array_equal(labels, y_test.to_numpy(dtype=np.int64)):
    raise ValueError("Source test labels do not match split row order")
tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
metrics = {
    "fraud_precision": float(precision_score(labels, predictions, zero_division=0)),
    "fraud_recall": float(recall_score(labels, predictions)),
    "fraud_f1": float(f1_score(labels, predictions)),
    "weighted_f1": float(f1_score(labels, predictions, average="weighted")),
    "weighted_recall": float(recall_score(labels, predictions, average="weighted")),
    "accuracy": float(accuracy_score(labels, predictions)),
    "roc_auc": float(roc_auc_score(labels, scores)),
    "ap": float(average_precision_score(labels, scores)),
    "fpr": float(fp / (fp + tn)),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
}
if not np.isclose(metrics["weighted_f1"], test_f1) or not np.isclose(
    metrics["accuracy"], test_accuracy
):
    raise ValueError("Source metrics disagree with saved predictions")
pd.DataFrame(
    {
        "row_index": y_test.index,
        "y_true": labels,
        "score": scores,
        "prediction_argmax": predictions,
    }
).to_csv("test_predictions.csv.gz", index=False)
Path("test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")


Writing D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/save_artifacts.py


In [12]:
%%writefile -a "{POSTPROCESS_FILE}"
Path("classification_report.json").write_text(
    json.dumps(
        classification_report(labels, predictions, output_dict=True, zero_division=0),
        indent=2,
    ),
    encoding="utf-8",
)

splits = [
    ("train", X_train_df, y_train_df),
    ("validation", X_val_df, y_val_df),
    ("random_test", X_test, y_test),
]
split_table = pd.concat(
    [
        pd.DataFrame(
            {
                "partition": name,
                "row_index": x.index,
                "y_true": y.to_numpy(dtype=np.int64),
            }
        )
        for name, x, y in splits
    ]
)
if split_table.row_index.duplicated().any() or len(split_table) != EXPECTED_ROWS:
    raise ValueError("Overlapping or incomplete original splits")
split_table.to_csv("split_indices.csv.gz", index=False)
preprocessing = {
    "numeric_columns": num_cols,
    "categorical_columns": cat_cols,
    "fit_rows": len(X_train),
    "imputer_statistics": num_pipeline.named_steps["imputer"].statistics_.tolist(),
    "scaler_mean": num_pipeline.named_steps["scaler"].mean_.tolist(),
    "scaler_scale": num_pipeline.named_steps["scaler"].scale_.tolist(),
    "cat_to_index": cat_to_index,
}
Path("preprocessing.json").write_text(
    json.dumps(preprocessing, indent=2), encoding="utf-8"
)


Appending to D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/save_artifacts.py


In [13]:
%%writefile -a "{POSTPROCESS_FILE}"
Path("source_results.json").write_text(
    json.dumps(
        {
            "hpo_trials": len(replay_trials),
            "best_params": best_params,
            "best_validation_f1_weighted": float(best_f1),
            "fixed_hyperparams": hyperparams,
            "class_weights": class_weights.tolist(),
            "source_test_f1_weighted": float(test_f1),
            "source_test_accuracy": float(test_accuracy),
            "split_rows": {name: len(x) for name, x, y in splits},
        },
        default=lambda value: (
            value.item() if isinstance(value, np.generic) else str(value)
        ),
        indent=2,
    ),
    encoding="utf-8",
)
print("ORIGINAL_GRU_ARTIFACTS_READY")
print(json.dumps(metrics, indent=2))


Appending to D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/save_artifacts.py


### 4.3. Thực thi và ghi output từng cell

Chạy trong tiến trình Python/IPython riêng. Log hiện ngay trong notebook 07; notebook chứa code tác giả được lưu sau từng cell.


In [14]:
%%writefile "{RUNNER_FILE}"
import contextlib
import io
import sys
import traceback
from pathlib import Path

import nbformat
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display
from IPython.utils.capture import capture_output


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


path = Path(sys.argv[1]).resolve()
notebook = nbformat.read(path, as_version=4)
shell = InteractiveShell.instance()
shell.user_ns["display"] = display
count = 0


Writing D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/execute_notebook.py


In [15]:
%%writefile -a "{RUNNER_FILE}"
for cell in notebook.cells:
    if cell.cell_type != "code":
        continue
    count += 1
    stdout = io.StringIO()
    stderr = io.StringIO()
    print("BEGIN_CELL", cell.metadata.get("author_cell", "evidence"), flush=True)
    with capture_output(stdout=False, stderr=False, display=True) as rich:
        with contextlib.redirect_stdout(Tee(sys.stdout, stdout)):
            with contextlib.redirect_stderr(Tee(sys.stderr, stderr)):
                result = shell.run_cell(cell.source, store_history=True)
    cell.execution_count = count
    cell.outputs = []
    for name, stream in [("stdout", stdout), ("stderr", stderr)]:
        if stream.getvalue():
            cell.outputs.append(
                nbformat.v4.new_output("stream", name=name, text=stream.getvalue())
            )
    for output in rich.outputs:
        cell.outputs.append(
            nbformat.v4.new_output(
                "display_data", data=output.data, metadata=output.metadata
            )
        )
    error = result.error_before_exec or result.error_in_exec
    if error:
        cell.outputs.append(
            nbformat.v4.new_output(
                "error",
                ename=type(error).__name__,
                evalue=str(error),
                traceback=traceback.format_exception(
                    type(error), error, error.__traceback__
                ),
            )
        )
    nbformat.write(notebook, path)
    if error:
        raise RuntimeError("Author notebook execution failed") from error
    print("END_CELL", cell.metadata.get("author_cell", "evidence"), flush=True)
print("EXECUTED_NOTEBOOK_SAVED", path, flush=True)


Appending to D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/execute_notebook.py


### 4.4. Kiểm chứng kết quả độc lập

Kiểm tra nguồn, 10 trial, 77 epoch, checkpoint, split và metric từ predictions.


In [16]:
%%writefile "{AUDIT_FILE}"
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

PROOF_DIR = None  # Optional: path to extracted proof directory.
here = Path.cwd().resolve()
root = (
    Path(PROOF_DIR).resolve()
    if PROOF_DIR
    else next(
        (p for p in [here, *here.parents] if (p / "author_run.json").is_file()), None
    )
)
if root is None:
    raise FileNotFoundError("Open the extracted proof directory or set PROOF_DIR")
info = json.loads((root / "author_run.json").read_text(encoding="utf-8"))
assert info["state"] == "completed"
assert info["commit"] == "6e8252cb9080378905b2aae30033c5946b72335b"
original_path = root / "evidence/author_original.ipynb"
assert (
    hashlib.sha256(original_path.read_bytes()).hexdigest()
    == info["source_notebook_sha256"]
)
original = json.loads(original_path.read_text(encoding="utf-8"))
executed = json.loads((root / "author_gru_executed.ipynb").read_text(encoding="utf-8"))
mapping = json.loads((root / "evidence/cell_mapping.json").read_text(encoding="utf-8"))


Writing D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/verify_results.py


In [17]:
%%writefile -a "{AUDIT_FILE}"
for row in mapping:
    source = "".join(original["cells"][row["author_cell"]]["source"])
    cell = next(
        c
        for c in executed["cells"]
        if c.get("metadata", {}).get("author_cell") == row["author_cell"]
    )
    effective = "".join(cell["source"])
    assert hashlib.sha256(source.encode()).hexdigest() == row["original_sha256"]
    assert hashlib.sha256(effective.encode()).hexdigest() == row["executed_sha256"]
    assert row["modified"] == (source != effective)
    assert cell["execution_count"] is not None
assert not [
    o
    for c in executed["cells"]
    for o in c.get("outputs", [])
    if o["output_type"] == "error"
]
assert sorted(row["author_cell"] for row in mapping if row["modified"]) == [0, 29]

trials = json.loads((root / "hpo_trials.json").read_text(encoding="utf-8"))
result = json.loads((root / "source_results.json").read_text(encoding="utf-8"))
assert len(trials) == result["hpo_trials"] == 10
rng = np.random.RandomState(1234)
for number, trial in enumerate(trials, 1):
    params = {
        "embedding_dim": rng.uniform(150, 250),
        "hidden_dim": rng.uniform(256, 768),
        "lr": 10 ** rng.uniform(np.log10(1e-4), np.log10(1e-3)),
        "epochs": int(rng.randint(5, 10)),
    }
    assert trial["trial"] == number
    assert all(
        np.isclose(trial["hyperparams"][key], value) for key, value in params.items()
    )
best = max(trials, key=lambda row: row["validation_f1_weighted"])
assert best["hyperparams"] == result["best_params"]
assert np.isclose(best["validation_f1_weighted"], result["best_validation_f1_weighted"])
epochs = re.findall(
    r"^Epoch (\d+)/(\d+), Loss: ([0-9.]+)",
    (root / "training.log").read_text(encoding="utf-8"),
    flags=re.MULTILINE,
)
expected_epoch_count = sum(t["hyperparams"]["epochs"] for t in trials) + 8
assert len(epochs) == expected_epoch_count == 77


Appending to D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/verify_results.py


In [18]:
%%writefile -a "{AUDIT_FILE}"
expected_epochs = [
    (epoch, trial["hyperparams"]["epochs"])
    for trial in trials
    for epoch in range(1, trial["hyperparams"]["epochs"] + 1)
] + [(epoch, 8) for epoch in range(1, 9)]
assert [(int(a), int(b)) for a, b, _ in epochs] == expected_epochs
assert result["fixed_hyperparams"] == {
    "embedding_dim": 186.88240060019746,
    "hidden_dim": 733.7677322150511,
    "lr": 0.000448103301026688,
    "epochs": 8,
}
assert (root / "GRU_Binary.pth").stat().st_size > 0
assert (root / "best_gru_model.pth").stat().st_size > 0

splits = pd.read_csv(root / "split_indices.csv.gz")
assert len(splits) == info["data_rows"] and not splits.row_index.duplicated().any()
assert splits.groupby("partition").size().to_dict() == result["split_rows"]
pred = pd.read_csv(root / "test_predictions.csv.gz")
test = splits[splits.partition == "random_test"]
assert np.array_equal(test.row_index, pred.row_index)
assert np.array_equal(test.y_true, pred.y_true)
assert len(pred) == int(info["data_rows"] * 0.2)
assert not pred.isna().any().any() and pred.score.between(0, 1).all()
y = pred.y_true.to_numpy(dtype=np.int64)
score = pred.score.to_numpy(dtype=np.float32)
label = pred.prediction_argmax.to_numpy(dtype=np.int64)
assert np.array_equal(label, (score > 0.5).astype(np.int64))
tn, fp, fn, tp = confusion_matrix(y, label, labels=[0, 1]).ravel()
values = {
    "fraud_precision": precision_score(y, label, zero_division=0),
    "fraud_recall": recall_score(y, label),
    "fraud_f1": f1_score(y, label),
    "weighted_f1": f1_score(y, label, average="weighted"),
    "weighted_recall": recall_score(y, label, average="weighted"),
    "accuracy": accuracy_score(y, label),
    "roc_auc": roc_auc_score(y, score),
    "ap": average_precision_score(y, score),
    "fpr": fp / (fp + tn),
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
}
saved = json.loads((root / "test_metrics.json").read_text(encoding="utf-8"))
assert all(
    np.isclose(saved[key], value, rtol=1e-7, atol=1e-8) for key, value in values.items()
)


Appending to D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/verify_results.py


In [19]:
%%writefile -a "{AUDIT_FILE}"
hash_path = root / "sha256.json"
if hash_path.is_file():
    hashes = json.loads(hash_path.read_text(encoding="utf-8"))
    for name, digest in hashes.items():
        assert (
            hashlib.sha256((root / name.replace(chr(92), "/")).read_bytes()).hexdigest()
            == digest
        )
    print("Verified SHA-256 entries:", len(hashes))
print("ORIGINAL_GRU_PROOF_VERIFIED: HPO10 + fixed8 = 77 epochs")
print(json.dumps(saved, indent=2))


Appending to D:/UIT/DS317/BAF_project/results/runs/author_gru_original_20261027_205457_a9c674ea/evidence/verify_results.py


## 5. Ghép các cell gốc thành notebook thực thi


In [20]:
OBSERVER = Path(OBSERVER_FILE).read_text(encoding="utf-8")
POSTPROCESS = Path(POSTPROCESS_FILE).read_text(encoding="utf-8")
AUDIT_SOURCE = Path(AUDIT_FILE).read_text(encoding="utf-8")

for name, source in [
    ("observer", OBSERVER),
    ("postprocess", POSTPROCESS),
    ("runner", Path(RUNNER_FILE).read_text(encoding="utf-8")),
    ("audit", AUDIT_SOURCE),
]:
    compile(source, name, "exec")


In [21]:
AUTHOR_CELLS = [0, 2, 3, 27, 28, 29, 30, 31, 33]

execution = nbformat.v4.new_notebook()

execution.metadata["kernelspec"] = {
    "name": "python3",
    "display_name": "Python 3",
    "language": "python",
}

execution.cells.append(
    nbformat.v4.new_markdown_cell(
        "# Direct execution of author GRU cells\n\nPinned commit: "
        + COMMIT
        + (
            ". Source cell IDs are zero-based. "
            "HPO10 then fixed recipe, original RNG order."
        )
    )
)

setup = (
    "from pathlib import Path\nimport json\nDATA_PATH = Path("
    + repr(str(DATA_PATH))
    + ")\nEXPECTED_ROWS = "
    + repr(EXPECTED_ROWS)
)

execution.cells.append(nbformat.v4.new_code_cell(setup))

changes = []

mapping = []


### Giữ nguyên cell tác giả và ghi rõ hai chỉnh sửa

Code lấy từ Git blob. Chỉ cell 0 và 29 được sửa như đã mô tả.
Những cell model, train và test còn lại được giữ nguyên văn.


In [22]:
for index in AUTHOR_CELLS:
    source = "".join(original["cells"][index]["source"])
    effective = source
    if index == 0:
        old = "df = pd.read_csv('E:/python2222/data/Base.csv')"
        if effective.count(old) != 1:
            raise ValueError(
                "Unknown author CSV loader; review source before patching."
            )
        effective = effective.replace(old, "df = pd.read_csv(DATA_PATH)")
    if index == 29:
        marker = "# Start timer"
        if effective.count(marker) != 1:
            raise ValueError("Unknown HPO cell layout.")
        effective = effective.replace(marker, OBSERVER + marker)
    if effective != source:
        changes.extend(
            difflib.unified_diff(
                source.splitlines(True),
                effective.splitlines(True),
                fromfile=f"original/cell_{index}",
                tofile=f"executed/cell_{index}",
            )
        )
    mapping.append(
        {
            "author_cell": index,
            "original_sha256": hashlib.sha256(source.encode()).hexdigest(),
            "executed_sha256": hashlib.sha256(effective.encode()).hexdigest(),
            "modified": effective != source,
        }
    )
    execution.cells.append(nbformat.v4.new_markdown_cell(f"## Author cell {index}"))
    code_cell = nbformat.v4.new_code_cell(effective)
    code_cell.metadata["author_cell"] = index
    execution.cells.append(code_cell)
    if index == 0:
        execution.cells.append(
            nbformat.v4.new_code_cell(
                "if len(df) != EXPECTED_ROWS "
                "or set(df.fraud_bool.unique()) != {0, 1}:\n"
                "    raise ValueError('Wrong input row count or labels')\n"
            )
        )


### Lưu notebook chuẩn bị thực thi


In [23]:
execution.cells.append(
    nbformat.v4.new_markdown_cell("## Save evidence after original test cell")
)

execution.cells.append(nbformat.v4.new_code_cell(POSTPROCESS))

write_json(EVIDENCE / "cell_mapping.json", mapping)

(EVIDENCE / "source_changes.diff").write_text("".join(changes), encoding="utf-8")

EXECUTED_PATH = RUN_DIR / "author_gru_executed.ipynb"

nbformat.write(execution, EXECUTED_PATH)


### Ghi nhận môi trường và phạm vi của lượt chạy


In [24]:
versions = {
    name: importlib.metadata.version(name)
    for name in [
        "numpy",
        "pandas",
        "scikit-learn",
        "scipy",
        "torch",
        "joblib",
        "nbformat",
        "ipython",
        "ipykernel",
        "tqdm",
    ]
}

run_info = {
    "state": "prepared",
    "repo": REPO_URL,
    "commit": COMMIT,
    "source_notebook_sha256": AUTHOR_SHA256,
    "data_sha256": EXPECTED_DATA_SHA256,
    "data_rows": EXPECTED_ROWS,
    "author_cells": AUTHOR_CELLS,
    "modified_author_cells": [0, 29],
    "scope": (
        "GRU original HPO10 plus original fixed recipe and test; "
        "no classical/importance/CI replay"
    ),
    "rng": "Original seed1234 in cell29; no reset after HPO; original DataLoader order",
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "packages": versions,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_build": torch.version.cuda,
    "progress_change": "TQDM_DISABLE=1 only; original epoch print retained",
    "preprocessing_note": (
        "Original fit on outer80%, includes inner validation features; "
        "random split includes month7"
    ),
    "test_model": (
        "Fixed model from author cell30, as author cell33; " "not automatic HPO winner"
    ),
}

write_json(RUN_DIR / "author_run.json", run_info)

print("Author cells:", AUTHOR_CELLS)

print("Modified cells:", [row["author_cell"] for row in mapping if row["modified"]])

print("Execution notebook:", EXECUTED_PATH)


Author cells: [0, 2, 3, 27, 28, 29, 30, 31, 33]
Modified cells: [0, 29]
Execution notebook: D:\UIT\DS317\BAF_project\results\runs\author_gru_original_20261027_205457_a9c674ea\author_gru_executed.ipynb


## 6. Chạy GRU gốc: HPO rồi fixed recipe

**Cell tiếp theo bắt đầu training.** Giữ máy thức và theo dõi log epoch/trial.
Không reset seed sau HPO. Model test là model fixed ở cell 30 như code tác giả.
Notebook training thật được lưu sau từng cell, không phụ thuộc Ctrl+S.


In [25]:
if run_info["state"] != "prepared":
    raise RuntimeError("Lượt này đã chạy. Restart Kernel + Run All để tạo run mới.")

runner_path = Path(RUNNER_FILE)

child_env = os.environ.copy()

child_env["TQDM_DISABLE"] = "1"

child_env["PYTHONUNBUFFERED"] = "1"

child_env["PYTHONIOENCODING"] = "utf-8"

run_info["state"] = "running"

run_info["execution_engine"] = (
    "Fresh Python process with IPython InteractiveShell; no native Jupyter kernel"
)

run_info["started"] = datetime.datetime.now().astimezone().isoformat()

write_json(RUN_DIR / "author_run.json", run_info)

process = None

started = time.perf_counter()


In [26]:
try:
    with (RUN_DIR / "training.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            [sys.executable, "-u", str(runner_path), str(EXECUTED_PATH)],
            cwd=RUN_DIR,
            env=child_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(
            (
                f"Author execution failed ({return_code}); "
                "xem training.log và notebook partial."
            )
        )
    run_info["state"] = "completed"
except BaseException as error:
    run_info["state"] = "failed_or_interrupted"
    run_info["error"] = repr(error)
    raise
finally:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    run_info["elapsed_seconds"] = time.perf_counter() - started
    run_info["finished"] = datetime.datetime.now().astimezone().isoformat()
    write_json(RUN_DIR / "author_run.json", run_info)

print("Original pipeline completed (minutes):", run_info["elapsed_seconds"] / 60)


BEGIN_CELL evidence
END_CELL evidence
BEGIN_CELL 0

Label distribution:
fraud_bool
0    988971
1     11029
Name: count, dtype: int64
Positive rate = 0.011029

First 5 rows:

Note: We will use variables X, y, df as the main data objects in the following steps.
END_CELL 0
BEGIN_CELL evidence
END_CELL evidence
BEGIN_CELL 2
Final num_cols count: 16
Final cat_cols count: 13
END_CELL 2
BEGIN_CELL 3
Dataset split completed. Train shape: (800000, 31) Test shape: (200000, 31)
Train positives: 8823  Test positives: 2206
Transformed types: <class 'numpy.ndarray'> <class 'numpy.ndarray'>
Transformed shapes (train/test): (800000, 59) (200000, 59)
Number of post-transform features (approx): 59
First 30 feature names: ['income', 'name_email_similarity', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'credit_risk_score', 'bank_months

## 7. Đối chiếu điểm được báo cáo trong paper

Đây là các điểm đã làm tròn ở Tables 2–3. Bảng hiển thị chênh lệch;
không tự chỉnh threshold để đuổi theo số paper hoặc kết luận khớp tuyệt đối.


In [27]:
PAPER = {
    "fraud_precision": 0.05,
    "fraud_recall": 0.78,
    "fraud_f1": 0.09,
    "weighted_f1": 0.8971,
    "weighted_recall": 0.83,
    "roc_auc": 0.8800,
}
write_json(
    RUN_DIR / "paper_reference.json",
    {
        "url": "https://www.mdpi.com/2079-3197/13/12/290",
        "doi": "10.3390/computation13120290",
        "locator": "Tables2/3: GRU row",
        "reported_rounded_points": PAPER,
        "comparison": (
            "Point comparison only; " "no assertion of exact numerical reproduction"
        ),
    },
)
saved = json.loads((RUN_DIR / "test_metrics.json").read_text(encoding="utf-8"))
import pandas as pd
from IPython.display import display

paper_comparison = pd.DataFrame(
    [
        {
            "metric": name,
            "paper": value,
            "this_run": saved[name],
            "difference": saved[name] - value,
        }
        for name, value in PAPER.items()
    ]
)
paper_comparison.to_csv(RUN_DIR / "paper_comparison.csv", index=False)
display(paper_comparison)
print("Scope:", run_info["scope"])


,metric,paper,this_run,difference
0,fraud_precision,0.0500,0.036935,-0.013065
1,fraud_recall,0.7800,0.845422,0.065422
2,fraud_f1,0.0900,0.070778,-0.019222
3,weighted_f1,0.8971,0.850304,-0.046796
4,weighted_recall,0.8300,0.755150,-0.074850
5,roc_auc,0.8800,0.882222,0.002222


Scope: GRU original HPO10 plus original fixed recipe and test; no classical/importance/CI replay


## 8. Kiểm chứng và xuất proof một lần

Proof giữ notebook gốc, diff, notebook training có output thật, log, HPO,
model và predictions. verify.ipynb là notebook kiểm chứng riêng, không train lại.


In [28]:
audit_text = AUDIT_SOURCE.replace(
    "PROOF_DIR = None", "PROOF_DIR = " + repr(str(RUN_DIR)), 1
)

audit_stdout = io.StringIO()

with contextlib.redirect_stdout(audit_stdout):
    exec(compile(audit_text, "<original GRU audit>", "exec"), {"__name__": "__audit__"})

print(audit_stdout.getvalue())

verify = nbformat.v4.new_notebook(
    cells=[
        nbformat.v4.new_markdown_cell(
            (
                "# Verify direct author GRU run\n\n"
                "Run All after extracting. No GPU/raw data needed. "
                "Set PROOF_DIR if necessary."
            )
        ),
        nbformat.v4.new_code_cell(
            AUDIT_SOURCE,
            execution_count=1,
            outputs=[
                nbformat.v4.new_output(
                    "stream", name="stdout", text=audit_stdout.getvalue()
                )
            ],
        ),
    ]
)

nbformat.write(verify, RUN_DIR / "verify.ipynb")

write_json(
    RUN_DIR / "export_status.json",
    {"state": "completed", "training_performed_by_export": False},
)

hashes = {
    str(p.relative_to(RUN_DIR)): sha256_file(p)
    for p in RUN_DIR.rglob("*")
    if p.is_file() and p.name != "sha256.json"
}

write_json(RUN_DIR / "sha256.json", hashes)


ORIGINAL_GRU_PROOF_VERIFIED: HPO10 + fixed8 = 77 epochs
{
  "fraud_precision": 0.0369350813958094,
  "fraud_recall": 0.8454215775158658,
  "fraud_f1": 0.07077798861480077,
  "weighted_f1": 0.8503039178974042,
  "weighted_recall": 0.75515,
  "accuracy": 0.75515,
  "roc_auc": 0.8822224491994386,
  "ap": 0.1573626938623287,
  "fpr": 0.2458568005096211,
  "tn": 149165,
  "fp": 48629,
  "fn": 341,
  "tp": 1865
}



In [29]:
PROOF_ROOT = ROOT / "results/proofs"

PROOF_ROOT.mkdir(parents=True, exist_ok=True)

archive = PROOF_ROOT / (RUN_DIR.name + "_proof_" + uuid.uuid4().hex[:8] + ".zip")

with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
    for path in RUN_DIR.rglob("*"):
        if path.is_file():
            bundle.write(path, Path(RUN_DIR.name) / path.relative_to(RUN_DIR))

print("Proof ZIP:", archive)

print("Original training notebook:", EXECUTED_PATH)


Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\author_gru_original_20261027_205457_a9c674ea_proof_410a8e84.zip
Original training notebook: D:\UIT\DS317\BAF_project\results\runs\author_gru_original_20261027_205457_a9c674ea\author_gru_executed.ipynb


Khi hoàn tất, lưu notebook 07 và giữ ZIP proof được in ở cell cuối.
Phần code gốc nằm trong author_gru_executed.ipynb của proof.
